# Świadek statystyczny: NSP 2021 i obwody głosowania

Dane: ludność z NSP 2021 w kwadratach 125 m (GUS) i obwody głosowania z wyborów prezydenckich 2025 (PKW). Świadek jest deterministyczny: mówi, ile osób mieszkało w 2021 roku w kwadracie budynku i ile z tego przypada na sam budynek.

In [ ]:
%config InlineBackend.figure_formats = ["jpeg"]
%config InlineBackend.print_figure_kwargs = {"bbox_inches": "tight", "pil_kwargs": {"quality": 88}}

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, LogNorm
from matplotlib.patches import Patch
from mpl_toolkits.axes_grid1.anchored_artists import AnchoredSizeBar
from shapely.geometry import box

from agenci.statystyka import obwody_adresu, przydzial_nsp, wczytaj_nsp, wczytaj_obwody
from ortofoto.arkusze import czytaj, zasieg
from ortofoto.budynki import wczytaj_budynki, wybierz

plt.rcParams.update({"figure.dpi": 100, "font.size": 9, "axes.titlesize": 10,
                     "axes.spines.top": False, "axes.spines.right": False})
KOLORY = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]  # kategorie, zawsze w tej kolejności
NIEBIESKI = LinearSegmentedColormap.from_list("niebieski", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
TUSZ, NIEMIESZKALNE = "#333333", "#dcdcdc"


def skala(ax, metry, loc="lower right"):
    pasek = AnchoredSizeBar(ax.transData, metry, f"{metry} m", loc, pad=0.4, borderpad=0.5, sep=3,
                            frameon=True, size_vertical=metry / 30)
    pasek.patch.set_alpha(0.8)
    ax.add_artist(pasek)


def tys(x):
    return f"{x:,.0f}".replace(",", " ")


ARKUSZ = zasieg()
bud_miasta, kwadraty = przydzial_nsp(wczytaj_nsp())  # świadek dla całego miasta
miasto = kwadraty[kwadraty.index.isin(bud_miasta.komorka.dropna().unique())]  # kwadraty z budynkiem Wrocławia
bud = wczytaj_budynki().join(bud_miasta[["komorka", "ludnosc_komorki", "udzial", "przydzial_2021", "pow_kondygnacji"]])
obwody = wczytaj_obwody()

## 1. NSP 2021 w siatce 125 m

Kod komórki GUS zawiera współrzędne jej narożnika (EPSG:3035), więc kwadraty budujemy z samego kodu. Liczymy tylko kwadraty, w których stoi choć jeden budynek Wrocławia z BDOT10k.

In [ ]:
fig, axs = plt.subplot_mosaic([["mapa", "hist"]], figsize=(15, 6.2), layout="constrained", width_ratios=[2.2, 1])
ax = axs["mapa"]
miasto.plot(ax=ax, column="ludnosc", cmap=NIEBIESKI, norm=LogNorm(1, miasto.ludnosc.max()), linewidth=0,
            legend=True, legend_kwds={"label": "osoby w kwadracie (skala log.)", "shrink": 0.7})
ax.plot(*ARKUSZ.exterior.xy, color=KOLORY[1], lw=2)
ax.annotate("arkusz ortofotomapy", xy=ARKUSZ.exterior.coords[2], xytext=(15, 15), textcoords="offset points",
            color=KOLORY[1], fontsize=9, arrowprops=dict(arrowstyle="-", color=KOLORY[1]))
ax.set_title(f"{tys(len(miasto))} kwadratów, {tys(miasto.ludnosc.sum())} osób")
ax.set_axis_off()
skala(ax, 2000)

h = axs["hist"]
h.hist(miasto.ludnosc, bins=np.logspace(0, np.log10(miasto.ludnosc.max()), 30), color=KOLORY[0], edgecolor="white")
h.set_xscale("log")
h.set(title="Ludność kwadratu", xlabel="osoby w kwadracie 125 × 125 m", ylabel="liczba kwadratów")
plt.show()

## 2. Trzy źródła, trzy sumy

Spis, rejestr wyborców i ekspertyza dla miasta podają różną liczbę mieszkańców Wrocławia. Dlatego oceniamy ranking budynków, a nie zgodność z jedną sumą.

In [ ]:
stale = obwody[obwody["Typ obwodu"] == "stały"]
sumy = pd.Series({"NSP 2021 (GUS), kwadraty miasta": miasto.ludnosc.sum(),
                  "mieszkańcy obwodów, 2025 (PKW)": obwody["Mieszkańcy"].sum(),
                  "wyborcy w obwodach, 2025 (PKW)": obwody["Wyborcy"].sum(),
                  "ekspertyza dla miasta (plan projektu)": 893_506})[::-1]
fig, ax = plt.subplots(figsize=(11, 3.2), layout="constrained")
ax.barh(sumy.index, sumy.values / 1000, color=KOLORY[0], height=0.6)
nsp = sumy["NSP 2021 (GUS), kwadraty miasta"]
for y, v in enumerate(sumy.values):
    ax.text(v / 1000, y, f"  {tys(v)}  ({v / nsp - 1:+.0%} wobec NSP)" if v != nsp else f"  {tys(v)}",
            va="center", fontsize=8.5, color=TUSZ)
ax.set(xlabel="tys. osób", xlim=(0, 1150))
ax.tick_params(axis="y", length=0)
plt.show()

## 3. Ludność kwadratu rozdzielona na budynki

Ludność kwadratu dzielimy między budynki mieszkalne i eksploatowane z BDOT10k proporcjonalnie do powierzchni kondygnacji (powierzchnia zabudowy × kondygnacje). Tak powstaje baseline `nsp_2021`.

In [ ]:
kw_ark = kwadraty[kwadraty.intersects(ARKUSZ)]
img, ext = czytaj("N", ARKUSZ.bounds, 1.0)
gora = bud.przydzial_2021[bud.przydzial_2021 > 0].quantile(0.98)

fig, axs = plt.subplot_mosaic([["mapa", "rozrzut"]], figsize=(16, 6.4), layout="constrained", width_ratios=[2.3, 1])
ax = axs["mapa"]
ax.imshow(255 - (255 - img.mean(2)) * 0.4, extent=ext, cmap="gray", vmin=0, vmax=255)
bud[bud.przydzial_2021.fillna(0) == 0].plot(ax=ax, color=NIEMIESZKALNE, edgecolor="white", linewidth=0.3)
bud[bud.przydzial_2021 > 0].plot(ax=ax, column="przydzial_2021", cmap=NIEBIESKI, vmin=0, vmax=gora, edgecolor="white",
                                 linewidth=0.3, legend=True,
                                 legend_kwds={"label": "osoby w budynku (NSP 2021, skala ucięta na 98. percentylu)",
                                              "shrink": 0.75})
kw_ark.boundary.plot(ax=ax, color="#7a7a7a", linewidth=0.6)
for kod, k in kw_ark.iterrows():
    p = k.geometry.centroid
    if ARKUSZ.contains(p) and k.ludnosc > 0:
        ax.text(p.x, p.y, tys(k.ludnosc), ha="center", va="center", fontsize=6.5, color=TUSZ,
                bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.7))
wiezowiec = bud.loc[wybierz(bud, osm_id=101207542).name]
ax.annotate(f"blok 12-kondygnacyjny: {wiezowiec.przydzial_2021:.0f} osób", xy=wiezowiec.geometry.centroid.coords[0],
            xytext=(-40, 60), textcoords="offset points", fontsize=8.5, color=TUSZ,
            arrowprops=dict(arrowstyle="-", color=TUSZ), bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="none"))
ax.legend(handles=[Patch(color=NIEMIESZKALNE, label="niemieszkalne albo nieeksploatowane")], loc="upper left", fontsize=8)
ax.set(title="Liczby w kwadratach: ludność z NSP 2021", xlim=ext[:2], ylim=ext[2:])
ax.set_axis_off()
skala(ax, 100)

m = bud[bud.przydzial_2021 > 0]
objetosc = m.geometry.area * m.h
r = objetosc.rank().corr(m.przydzial_2021.rank())
s = axs["rozrzut"]
s.scatter(objetosc, m.przydzial_2021, s=14, color=KOLORY[0], alpha=0.75, edgecolor="white", linewidth=0.3)
s.set(xscale="log", yscale="log", xlabel="objętość budynku [m³]", ylabel="osoby (NSP 2021 rozdzielone)",
      title=f"Ten sam budynek w dwóch baseline'ach (Spearman {r:.2f})")
plt.show()

Co świadek zwraca dla trzech budynków z poprzednich notatników:

In [ ]:
przyklady = {"blok 12 kond.": 101207542, "kamienica, Piastowska 36": 101211778, "segment bloku 5 kond.": 101189011}
pd.DataFrame({nazwa: {"kwadrat (NSP 2021)": bud.loc[wybierz(bud, osm_id=o).name, "ludnosc_komorki"],
                      "udział budynku w kwadracie": bud.loc[wybierz(bud, osm_id=o).name, "udzial"],
                      "osoby w budynku (2021)": bud.loc[wybierz(bud, osm_id=o).name, "przydzial_2021"],
                      "gęstość kwadratu [os./100 m²]": kwadraty.loc[bud.loc[wybierz(bud, osm_id=o).name, "komorka"],
                                                                   "gestosc_100m2"]}
              for nazwa, o in przyklady.items()}).round(2)

## 4. Gęstość zaludnienia: dane do kalibracji rubryki

Ile osób z 2021 roku przypada na 100 m² powierzchni kondygnacji mieszkalnych w kwadracie. Agent rubryki może z tego uczyć się gęstości dla typów zabudowy zamiast zgadywać je z wiedzy ogólnej.

In [ ]:
g = miasto[(miasto.ludnosc >= 20) & miasto.gestosc_100m2.notna()]
p10, p50, p90 = g.gestosc_100m2.quantile([0.1, 0.5, 0.9])
bez = miasto[miasto.bez_budynku]

fig, axs = plt.subplot_mosaic([["hist", "mapa"]], figsize=(16, 6), layout="constrained", width_ratios=[1, 1.6])
h = axs["hist"]
h.hist(g.gestosc_100m2.clip(upper=8), bins=40, color=KOLORY[0], edgecolor="white")
h.axvspan(p10, p90, color="#e6f0fb", zorder=0, label=f"10.-90. percentyl: {p10:.1f}-{p90:.1f}")
h.axvline(p50, color=TUSZ, lw=1.2, label=f"mediana: {p50:.2f}")
h.set(title=f"{tys(len(g))} kwadratów z co najmniej 20 osobami", xlabel="osoby na 100 m² powierzchni kondygnacji (ucięte na 8)",
      ylabel="liczba kwadratów")
h.legend(frameon=False)

ax = axs["mapa"]
g.plot(ax=ax, column="gestosc_100m2", cmap=NIEBIESKI, vmin=0, vmax=g.gestosc_100m2.quantile(0.98), linewidth=0,
       legend=True, legend_kwds={"label": "osoby na 100 m² (skala ucięta na 98. percentylu)", "shrink": 0.7})
bez.centroid.plot(ax=ax, color=KOLORY[1], markersize=6)
ax.legend(handles=[Patch(color=KOLORY[1], label=f"ludność bez budynku mieszkalnego w BDOT10k: "
                                               f"{len(bez)} kwadratów, {tys(bez.ludnosc.sum())} osób")],
          loc="lower left", fontsize=8)
ax.plot(*ARKUSZ.exterior.xy, color=TUSZ, lw=1.2)
ax.set_axis_off()
skala(ax, 2000)
plt.show()

## 5. Obwody głosowania PKW 2025

Każdy obwód ma liczbę mieszkańców, liczbę wyborców i opis granic w postaci listy ulic i numerów. To dane z 2025 roku, więc obejmują osiedla oddane po spisie.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 4.8), layout="constrained")
a = axs[0]
a.scatter(stale["Mieszkańcy"], stale["Wyborcy"], s=12, color=KOLORY[0], alpha=0.7, edgecolor="white", linewidth=0.3)
granica = max(stale["Mieszkańcy"].max(), stale["Wyborcy"].max())
a.plot([0, granica], [0, granica], color="#9a9a9a", ls="--", lw=1)
a.text(granica * 0.97, granica * 0.93, "wyborcy = mieszkańcy", ha="right", fontsize=8, color="#7a7a7a")
a.set(xlabel="mieszkańcy obwodu", ylabel="wyborcy w obwodzie",
      title=f"{len(stale)} obwodów stałych; obwody specjalne ({len(obwody) - len(stale)}) mają 0 mieszkańców")
b = axs[1]
b.hist(stale["Wyborcy"] / stale["Mieszkańcy"].replace(0, np.nan), bins=40, color=KOLORY[0], edgecolor="white")
b.set(xlabel="wyborcy / mieszkańcy", ylabel="liczba obwodów", title="Udział wyborców wśród mieszkańców obwodu")
b.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
plt.show()

## 6. Opis granic → budynki

Parser zamienia opis granic na reguły: ulica i zakresy numerów (z parzystością). Adres budynku trafia do obwodu, którego reguły go obejmują.

In [ ]:
przyklad = 67
print(obwody.loc[przyklad, "Opis granic"])
pd.DataFrame([(ulica, "cała ulica" if z is None else
               "; ".join(f"{od}-{do}" + {0: " parzyste", 1: " nieparzyste", None: ""}[p] for od, do, p in z))
              for ulica, z in obwody.loc[przyklad, "reguly"].items()], columns=["ulica", "numery"])

In [ ]:
z_adresem = bud[bud["addr:street"].notna()].copy()
z_adresem["obwody"] = [obwody_adresu(r["addr:street"], r["addr:housenumber"], obwody) for _, r in z_adresem.iterrows()]
z_adresem["obwod"] = z_adresem.obwody.map(lambda o: o[0] if len(o) == 1 else None)
numery = sorted(z_adresem.obwod.dropna().astype(int).unique())

fig, ax = plt.subplots(figsize=(14, 6.4), layout="constrained")
ax.imshow(255 - (255 - img.mean(2)) * 0.4, extent=ext, cmap="gray", vmin=0, vmax=255)
bud.plot(ax=ax, color=NIEMIESZKALNE, edgecolor="white", linewidth=0.3)
uchwyty = []
for nr, kolor in zip(numery, KOLORY):
    z_adresem[z_adresem.obwod == nr].plot(ax=ax, color=kolor, edgecolor="white", linewidth=0.3)
    uchwyty.append(Patch(color=kolor, label=f"obwód {nr}: {tys(obwody.loc[nr, 'Mieszkańcy'])} mieszk."))
uchwyty.append(Patch(color=NIEMIESZKALNE, label="bez adresu w OSM"))
jeden = (z_adresem.obwody.map(len) == 1).sum()
ax.legend(handles=uchwyty, loc="upper left", fontsize=8, framealpha=0.9)
ax.set(title=f"{jeden} z {len(z_adresem)} adresów trafia do dokładnie jednego obwodu; adres ma {len(z_adresem)} "
             f"z {len(bud)} budynków", xlim=ext[:2], ylim=ext[2:])
ax.set_axis_off()
skala(ax, 100)
plt.show()

Adres z OSM ma co czwarty budynek arkusza. Pełne pokrycie dadzą punkty adresowe Wrocławia (`Adresy.zip` z geoportalu miasta): wtedy każdy budynek dostanie obwód, a obwód sumę predykcji do porównania z liczbą mieszkańców z PKW.

## Miejsce w systemie

- Agent rubryki dostaje gęstość kwadratów do kalibracji, a agent bilansujący sumę z 2021 roku dla kwadratu budynku.
- `eksperymenty/nsp_2021.toml` to nowy baseline: dobry dla starej zabudowy, z definicji błędny dla osiedli oddanych po 2021 roku.
- Obwody PKW 2025 są oceną albo osobnym ramieniem ablacji, nigdy jednym i drugim w tym samym eksperymencie.